# 06 — RingTransfer: clear parallel paths + severe over-squashing

The synthetic bottleneck (notebook 04/05) is our own construction; LRGB Peptides (notebook 02) is real but has
only *mild* over-squashing. **RingTransfer** (Di Giovanni et al., 2023) is a *standard* over-squashing probe
with **two clear parallel paths**: a cycle of `n` nodes where a source must transmit its one-hot label to the
diametrically opposite target. Information can go the **two ways around the ring** — two arcs of length `n/2`
— and the walk multiplicity into the target grows like `4^{n/2}`, so over-squashing is severe and tunable.

**This is the regime where Walk Attention wins decisively**, confirming the regime-dependent story: as the
ring grows, one-hop GCN/GAT collapse while Walk Attention stays exact.

## 1. The ring and its two parallel arcs

The source (node 0) and target (node n/2) are connected by exactly two length-`n/2` paths — the two arcs.

In [ ]:
import numpy as np, scipy.sparse as sp, torch
from oversquash.data_ring import make_ring_transfer, ring_dataset

for n in [8, 12, 16, 20]:
    d = make_ring_transfer(n, num_classes=5, generator=torch.Generator().manual_seed(0))
    ei = d.edge_index.numpy()
    A = sp.csr_matrix((np.ones(ei.shape[1]), (ei[0], ei[1])), shape=(n, n)).toarray()
    g = n // 2
    Ag = np.linalg.matrix_power(A, g)
    print(f'ring n={n}: src->tgt length-{g} paths = {int(Ag[0, g])} (two arcs) | '
          f'total walks into target = {int(Ag[:, g].sum())} (~4^(n/2))')

## 2. Draw a ring

Source blue, target green; the two arcs between them are the parallel paths.

In [ ]:
from oversquash import viz
n = 10
edges = [(i, (i+1) % n) for i in range(n)] + [((i+1) % n, i) for i in range(n)]
pos = {i: (np.cos(2*np.pi*i/n), np.sin(2*np.pi*i/n)) for i in range(n)}
viz.draw_graph(edges, n, src=0, dst=n//2, pos=pos, title=f'RingTransfer n={n}: two arcs 0 -> {n//2}')

## 3. Train across ring sizes (severity grows with n)

We compare GCN / GAT (one-hop) vs Walk Attention (multi-hop), at `n/2` layers/ranges. Chance is `1/5 = 0.20`.
This is a short demo; the full sweep is `run_ring.py` (results in `results/tables/ring_transfer.csv`).

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.lrgb import AttachLRGBMasks, collate_lrgb
from oversquash.models import build_model
K = 5

def evaluate(m, loader, fwd):
    m.eval(); acc = []
    with torch.no_grad():
        for b in loader:
            lg = fwd(m, b); mm = b.root_mask
            acc.append((lg[mm].argmax(-1) == b.y[mm]).float().mean().item())
    return float(np.mean(acc))

def quick(name, n, epochs=30):
    torch.manual_seed(0); nl = n // 2
    tr = ring_dataset(n, 600, seed=0, num_classes=K); va = ring_dataset(n, 200, seed=1, num_classes=K)
    if name == 'walkattn':
        tf = AttachLRGBMasks(n_layers=nl); tr = [tf(d) for d in tr]; va = [tf(d) for d in va]
        trl = DataLoader(tr, batch_size=64, shuffle=True, collate_fn=collate_lrgb)
        val = DataLoader(va, batch_size=64, collate_fn=collate_lrgb)
        fwd = lambda m, b: m(b.x, b.edge_index, getattr(b,'batch',None), walk_masks=b.walk_masks)[0]
    else:
        trl = PyGLoader(tr, batch_size=64, shuffle=True); val = PyGLoader(va, batch_size=64)
        fwd = lambda m, b: m(b.x, b.edge_index, getattr(b,'batch',None))[0]
    m = build_model(name, tr[0].x.size(-1), 32, K, nl, heads=4)
    opt = torch.optim.Adam(m.parameters(), lr=5e-3)
    for e in range(epochs):
        m.train()
        for b in trl:
            opt.zero_grad(); F.cross_entropy(fwd(m,b), b.y, ignore_index=-100).backward(); opt.step()
    return evaluate(m, val, fwd)

for n in [10, 14]:
    print(f'ring n={n} (distance {n//2}):  '
          + '  '.join(f'{bb}={quick(bb, n):.3f}' for bb in ['gcn', 'gat', 'walkattn']))

## 4. Full result (from `run_ring.py`)

| ring `n` | distance | GCN | GAT | **Walk Attention** |
|---|---|---|---|---|
| 6 | 3 | 1.00 | 1.00 | **1.00** |
| 10 | 5 | 1.00 | 1.00 | **1.00** |
| 14 | 7 | 0.25 | 0.43 | **1.00** |
| 18 | 9 | 0.59 | 0.60 | **1.00** |

Small rings are easy for everyone. From `n=14` the one-hop baselines collapse (severe over-squashing) while
**Walk Attention stays at 1.000**. This is the clean win predicted by the theory: clear parallel paths + severe
squashing = the multi-hop, learned-weight operator is decisive. Contrast with LRGB (notebook 02), where the
squashing is mild and the models tie.